In [18]:
import numpy as np
from perf_func import time_stage, test_concurrency
import numba
from crs_estimate import c_sequence
from mib_decode import equalize_1ant, equalize_2ant, equalize_4ant, subblock_interleaver, ConvCoderNumba, CRC16, hamming_dist
import math

1. numpy

In [19]:
def pcfich_re_mask(N_rb, N_id):
    N_SC_RB = 12
    N_sc = N_rb * N_SC_RB
    crs_mod3 = (N_id % 6) % 3
    k_bar = (N_SC_RB // 2) * (N_id % (2 * N_rb))
    k_ind = np.zeros(16, dtype=int)
    m = 0
    for q in range(4):
        k_init = (k_bar + ((q * N_rb) // 2) * (N_SC_RB // 2)) % N_sc
        for i in range(6):
            if i % 3 != crs_mod3:
                k_ind[m] = (k_init + i) % N_sc
                m += 1
    return k_ind

In [20]:
class PCFICHDecoding:
    def __init__(self):
        self.EQUALIZE = {1: equalize_1ant, 2: equalize_2ant, 4: equalize_4ant}
        self.CFI_CODES = {
            1: np.array([0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1], dtype=np.uint8),
            2: np.array([1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0], dtype=np.uint8),
            3: np.array([1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1,0,1,1], dtype=np.uint8),
        }
        self._k_ind = None
        self._scramble_seq = None

    def config(self, N_id, N_rb, ns):
        self._k_ind = pcfich_re_mask(N_rb, N_id)
        c_init = (((ns // 2 + 1) * (2 * N_id + 1)) << 9) + N_id
        self._scramble_seq = c_sequence(32, c_init)

    def __call__(self, chunk):
        r = chunk.symbols[0, self._k_ind]
        H = chunk.H[:, 0, self._k_ind]
        eq = self.EQUALIZE[chunk.n_ant](r, H)
        bits = np.zeros(32, dtype=np.uint8)
        bits[0::2] = (eq.real < 0).astype(np.uint8)
        bits[1::2] = (eq.imag < 0).astype(np.uint8)
        bits = bits ^ self._scramble_seq
        best_cfi, best_dist = None, np.inf
        for cfi, code in self.CFI_CODES.items():
            d = np.sum(bits != code)
            if d < best_dist:
                best_dist, best_cfi = d, cfi
        chunk.cfi = best_cfi
        chunk.cfi_hamming = best_dist
        return chunk

In [21]:
def pdcch_reg_table(N_rb, N_id, cfi, n_ant, phich_res):
    nu_shift = N_id % 6
    crs_positions = {nu_shift, (3 + nu_shift) % 6}
    data_offsets_crs = [i for i in range(6) if i not in crs_positions]
    PHICH_RES_MAP = {'1/6': 1/6, '1/2': 1/2, '1': 1, '2': 2}
    all_REG = set(range(2 * N_rb))
    PCFI_REG = {(N_id % (2*N_rb) + (n*N_rb) // 2) % (2*N_rb) for n in range(4)}
    all_REG.difference_update(PCFI_REG)
    all_REG_vec = np.sort(np.array(list(all_REG)))
    n_0 = len(all_REG_vec)
    Ng = PHICH_RES_MAP[phich_res]
    N_group = int(np.ceil(Ng * N_rb / 8))
    for m in range(N_group):
        for i in range(3):
            n_i = (N_id + m + (i * n_0) // 3) % n_0
            all_REG.discard(all_REG_vec[n_i])
    reg_table = []
    for reg_idx in sorted(all_REG):
        k_start = 6 * reg_idx
        reg_table.append((k_start, 0, [k_start + off for off in data_offsets_crs]))
    for l in range(1, cfi):
        has_crs = (l == 1 and n_ant == 4)
        if has_crs:
            for reg_idx in range(2 * N_rb):
                k_start = 6 * reg_idx
                reg_table.append((k_start, l, [k_start + off for off in data_offsets_crs]))
        else:
            for reg_idx in range(3 * N_rb):
                k_start = 4 * reg_idx
                reg_table.append((k_start, l, [k_start, k_start+1, k_start+2, k_start+3]))
    reg_table.sort(key=lambda x: (x[0], x[1]))
    return reg_table, len(reg_table)


def pdcch_deinterleave(N_reg, N_id):
    pdcch_reg = np.arange(N_reg)
    pdcch_reg_cs = np.roll(pdcch_reg, N_id % N_reg)
    perm_table = subblock_interleaver(N_reg)
    pdcch_num_reg = np.zeros(N_reg, dtype=int)
    pdcch_num_reg[perm_table] = pdcch_reg_cs
    return pdcch_num_reg


def dci_1a_size(N_rb):
    N_RIV = math.ceil(math.log2(N_rb * (N_rb + 1) / 2))
    size_1a = 1 + 1 + N_RIV + 5 + 3 + 1 + 2 + 2
    if N_rb >= 50: size_1a += 1
    N_UL_hop = 2 if N_rb >= 50 else 1
    size_0 = 1 + N_UL_hop + N_RIV + 5 + 1 + 2 + 3 + 1
    return max(size_1a, size_0)


def conv_rate_dematch(e_bits, K):
    perm = subblock_interleaver(K)
    coded = np.zeros((3, K), dtype=np.uint8)
    for n in range(3):
        coded[n, perm] = e_bits[n * K:(n + 1) * K]
    return coded


def decode_dci_1a(dci_bits, N_rb):
    N_RIV = math.ceil(math.log2(N_rb * (N_rb + 1) / 2))
    TBS_TABLE = [
        [16,32,56,88,120,152],[24,56,88,144,176,208],[32,72,144,176,208,256],
        [40,104,176,208,256,328],[56,120,208,256,328,408],[72,144,224,328,424,504],
        [88,176,256,392,504,600],[104,224,328,472,584,712],[120,256,392,536,680,808],
        [136,296,456,616,776,936],
    ]
    def b2i(b):
        v = 0
        for x in b: v = (v << 1) | int(x)
        return v
    p = 2
    riv = b2i(dci_bits[p:p+N_RIV]); p += N_RIV
    i_mcs = b2i(dci_bits[p:p+5]); p += 5
    p += 4
    rv = b2i(dci_bits[p:p+2]); p += 2
    tpc = b2i(dci_bits[p:p+2]); p += 2
    L_crbs = riv // N_rb + 1
    rb_start = riv % N_rb
    if L_crbs > N_rb - rb_start:
        L_crbs = N_rb + 1 - riv // N_rb
        rb_start = N_rb - 1 - riv % N_rb
    n_prb = 3 if (tpc & 1) else 2
    tbs = TBS_TABLE[i_mcs][n_prb - 1]
    return {'rb_start': rb_start, 'L_crbs': L_crbs, 'Q_m': 2, 'rv': rv, 'tbs': tbs}

In [22]:
class PDCCHDecoding:
    def __init__(self):
        self.EQUALIZE = {1: equalize_1ant, 2: equalize_2ant, 4: equalize_4ant}
        self.fec = ConvCoderNumba([0o133, 0o171, 0o165])
        self.crc = CRC16()
        self._N_id = None; self._N_rb = None; self._n_ant = None
        self._dci_payload = None; self._scramble_seq = None
        self._cfi_table = [None]*4

    def config(self, N_id, N_rb, n_ant, phich_res, ns):
        self._N_id, self._N_rb, self._n_ant = N_id, N_rb, n_ant
        self._dci_payload = dci_1a_size(N_rb)
        max_bits = 0
        for cfi in [1, 2, 3]:
            reg_table, N_reg = pdcch_reg_table(N_rb, N_id, cfi, n_ant, phich_res)
            n_cce = N_reg // 9
            deinterleave = pdcch_deinterleave(N_reg, N_id)
            sym_indices = np.zeros(N_reg * 4, dtype=int)
            k_indices = np.zeros(N_reg * 4, dtype=int)
            for i, reg in enumerate(deinterleave):
                _, sym_l, k_list = reg_table[reg]
                for j, k in enumerate(k_list):
                    sym_indices[i*4+j] = sym_l
                    k_indices[i*4+j] = k
            self._cfi_table[cfi] = (N_reg, n_cce, sym_indices, k_indices)
            if N_reg * 8 > max_bits: max_bits = N_reg * 8
        c_init = ((ns // 2) << 9) + N_id
        self._scramble_seq = c_sequence(max_bits, c_init)

    def __call__(self, chunk):
        N_reg, n_cce, sym_idx, k_idx = self._cfi_table[chunk.cfi]
        r = chunk.symbols[sym_idx, k_idx]
        H = chunk.H[:, sym_idx, k_idx]
        x = self.EQUALIZE[self._n_ant](r, H)
        bits = np.zeros(N_reg * 8, dtype=np.uint8)
        bits[0::2] = (x.real < 0).astype(np.uint8)
        bits[1::2] = (x.imag < 0).astype(np.uint8)
        bits = bits ^ self._scramble_seq[:N_reg * 8]
        result = self._blind_search(bits, n_cce)
        if result:
            dci_bits, cost = result
            dci = decode_dci_1a(dci_bits, self._N_rb)
            chunk.dci_decoded = True; chunk.dci_cost = cost
            chunk.rb_start = dci['rb_start']; chunk.L_crbs = dci['L_crbs']
            chunk.Q_m = dci['Q_m']; chunk.rv = dci['rv']; chunk.tbs = dci['tbs']
        return chunk

    def _blind_search(self, pdcch_bits, n_cce):
        BITS_PER_CCE = 72; K = self._dci_payload + 16; tried = set()
        for L in [8, 4]:
            if n_cce < L: continue
            for m in range(2 if L == 8 else 4):
                start = L * (m % (n_cce // L))
                if start + L > n_cce or (L, start) in tried: continue
                tried.add((L, start))
                cce_bits = pdcch_bits[start*BITS_PER_CCE:(start+L)*BITS_PER_CCE]
                coded = conv_rate_dematch(cce_bits, K)
                decoded, cost = self.fec.decode(coded, hamming_dist)
                info = decoded[:self._dci_payload]
                parity = decoded[self._dci_payload:] ^ 1
                if self.crc(np.packbits(np.concatenate([info, parity]))) == 0:
                    return (info, cost)
        return None

In [23]:
def pdsch_crs_symbols(n_ant):
    return {0, 4, 7, 11} if n_ant <= 2 else {0, 1, 4, 7, 8, 11}

def pdsch_extract_indices(cfi, rb_start, L_crbs, crs_syms, nu_shift):
    k_start = rb_start * 12
    N_sc = L_crbs * 12
    k = np.arange(N_sc)
    mod6 = (k_start + k) % 6
    mask = (mod6 != nu_shift) & (mod6 != (nu_shift + 3) % 6)
    sym_list, k_list = [], []

    for l in range(cfi, 14):
        ks = np.arange(k_start, k_start + N_sc)[mask] if l in crs_syms else np.arange(k_start, k_start + N_sc)
        sym_list.append(np.full(len(ks), l, dtype=int)) 
        k_list.append(ks)

    return np.concatenate(sym_list), np.concatenate(k_list)


class PDSCHDecoding:
    def __init__(self):
        self.EQUALIZE = {1: equalize_1ant, 2: equalize_2ant, 4: equalize_4ant}
        self._n_ant = None 
        self._crs_syms = None 
        self._nu_shift = None

    def config(self, N_id, n_ant):
        self._n_ant = n_ant
        self._crs_syms = pdsch_crs_symbols(n_ant)
        self._nu_shift = N_id % 6

    def __call__(self, chunk):

        sym_idx, k_idx = pdsch_extract_indices(chunk.cfi, chunk.rb_start, chunk.L_crbs, self._crs_syms, self._nu_shift)
        r = chunk.symbols[sym_idx, k_idx]; H = chunk.H[:, sym_idx, k_idx]
        x = self.EQUALIZE[self._n_ant](r, H)
        chunk.pdsch_soft = np.zeros(len(x) * 2)
        chunk.pdsch_soft[0::2] = -x.real
        chunk.pdsch_soft[1::2] = -x.imag

        return chunk

In [24]:
class PDSCHDecoding:
    def __init__(self):
        self.EQUALIZE = {1: equalize_1ant, 2: equalize_2ant, 4: equalize_4ant}
        self._n_ant = None; self._crs_syms = None; self._nu_shift = None

    def config(self, N_id, n_ant):
        self._n_ant = n_ant; self._crs_syms = pdsch_crs_symbols(n_ant); self._nu_shift = N_id % 6

    def __call__(self, chunk):
        sym_idx, k_idx = pdsch_extract_indices(chunk.cfi, chunk.rb_start, chunk.L_crbs, self._crs_syms, self._nu_shift)
        r = chunk.symbols[sym_idx, k_idx]; H = chunk.H[:, sym_idx, k_idx]
        x = self.EQUALIZE[self._n_ant](r, H)
        chunk.pdsch_soft = np.zeros(len(x) * 2)
        chunk.pdsch_soft[0::2] = -x.real; chunk.pdsch_soft[1::2] = -x.imag
        return chunk

In [25]:
def pdsch_descramble(pdsch_soft, ns, N_id):
    c_init = (0xFFFF << 14) + ((ns // 2) << 9) + N_id
    c = c_sequence(len(pdsch_soft), c_init)
    return pdsch_soft * (1.0 - 2.0 * c)

def f1f2_table(K):
    table = {
        40:(3,10),48:(7,12),56:(19,42),64:(7,16),72:(7,18),80:(11,20),88:(5,22),96:(11,24),
        104:(7,26),112:(41,84),120:(103,90),128:(15,32),136:(9,34),144:(17,108),152:(9,38),
        160:(21,120),168:(101,84),176:(21,44),184:(57,46),192:(23,48),200:(13,50),208:(27,52),
        216:(11,36),224:(27,56),232:(85,58),240:(29,60),248:(33,62),256:(15,32),264:(17,198),
        272:(33,68),280:(103,210),288:(19,36),296:(19,74),304:(37,76),312:(19,78),320:(21,120),
        328:(21,82),336:(115,84),344:(193,86),352:(21,44),360:(133,90),368:(81,46),376:(45,94),
        384:(23,48),392:(243,98),400:(151,40),408:(155,102),416:(25,52),424:(51,106),432:(47,72),
        440:(91,110),448:(29,168),456:(29,114),464:(247,58),472:(29,118),480:(89,180),488:(91,122),
        496:(157,62),504:(55,84),512:(31,64),528:(17,66),544:(35,68),560:(227,420),576:(65,96),
        592:(19,74),608:(37,76),624:(41,234),640:(39,80),656:(185,82),672:(43,252),688:(21,86),
        704:(155,44),720:(79,120),736:(139,92),752:(23,94),768:(217,48),784:(25,98),800:(17,80),
    }
    return table[K]

def turbo_permutation(K):
    f1, f2 = f1f2_table(K)
    i = np.arange(K)
    return (f1 * i + f2 * i**2) % K

def turbo_permutation_inv(K):
    return np.argsort(turbo_permutation(K))

def turbo_len(transport_size):
    return transport_size + 24 + 4

def rate_matching_turbo(seq_len, col_perm_table, rv):
    C = 32; D = seq_len; DUMMY = D + 10000
    R = (D + C - 1) // C; K_pi = R * C; N_dummy = K_pi - D
    y = np.concatenate((DUMMY * np.ones(N_dummy, dtype=int), np.arange(seq_len)))
    M = np.reshape(y, (R, C))
    p = np.zeros_like(M)
    for n in range(C): p[:, n] = M[:, col_perm_table[n]]
    v = np.reshape(p.T, -1)
    k = np.arange(K_pi)
    pi_k = (col_perm_table[k // R] + C * (k % R) + 1) % K_pi
    v3 = y[pi_k]
    v1 = v.copy(); v2 = v.copy()
    v2[v2 < 10000] += D; v3[v3 < 10000] += 2 * D
    N_cb = 3 * K_pi; v_comb = np.zeros(N_cb, int)
    v_comb[:K_pi] = v1; v_comb[K_pi::2] = v2; v_comb[K_pi + 1::2] = v3
    k0 = int(R * (2 * np.ceil(N_cb / (8 * R)) * rv + 2))
    k, j = 0, 0; e = np.zeros(3 * D, int)
    while k < 3 * D:
        if v_comb[(k0 + j) % N_cb] != DUMMY: e[k] = v_comb[(k0 + j) % N_cb]; k += 1
        j += 1
    return e

def turbo_rate_dematch(pdsch_bits, coded_block_size, rv):
    col_perm = np.array([0,16,8,24,4,20,12,28,2,18,10,26,6,22,14,30,1,17,9,25,5,21,13,29,3,19,11,27,7,23,15,31])
    collect_len = 3 * coded_block_size
    perm_table = rate_matching_turbo(coded_block_size, col_perm, rv)
    coded_bits = np.zeros(collect_len)
    coded_bits[perm_table] = pdsch_bits[:collect_len]
    return coded_bits.reshape(3, coded_block_size)


# ---- BCJR / Turbo (NumPy) ----

def maxstar(a, b):
    result = np.empty_like(a)
    ainf = a == -np.inf; binf = b == -np.inf
    result[ainf] = b[ainf]; result[binf] = a[binf]
    noninf = ~ainf & ~binf
    an, bn = a[noninf], b[noninf]
    result[noninf] = np.maximum(an, bn) + np.log1p(np.exp(-np.abs(an - bn)))
    return result

def BCJR(Ru, R, Ap):
    T = Ru.size; nu = 3; Ns = 2**nu
    Gamma = np.full((T, Ns, Ns), -np.inf)
    A = np.full((T, Ns), -np.inf); A[0, 0] = 0
    B = np.full((T, Ns), -np.inf); B[-1, 0] = 0
    for r in range(Ns):
        for s in range(Ns):
            if r >> 1 == s & (2**(nu-1)-1):
                Gamma[:-nu, r, s] = 0
                feedback = (r ^ (r >> 1)) & 1
                newbit = (s >> (nu-1)) ^ feedback
                c0, c1 = newbit, (r ^ (r >> 2) ^ (s >> (nu-1))) & 1
                Gamma[:-nu,r,s] += (Ap if newbit==0 else -Ap) - np.log1p(np.exp(Ap if newbit==0 else -Ap))
                Gamma[:-nu,r,s] += (Ru[:-nu] if c0==0 else -Ru[:-nu]) - np.log1p(np.exp(Ru[:-nu] if c0==0 else -Ru[:-nu]))
                Gamma[:-nu,r,s] += (R[:-nu] if c1==0 else -R[:-nu]) - np.log1p(np.exp(R[:-nu] if c1==0 else -R[:-nu]))
            if r >> 1 == s:
                Gamma[-nu:, r, s] = 0
                feedback = (r ^ (r >> 1)) & 1; c0 = feedback; c1 = (r ^ (r >> 2)) & 1
                Gamma[-nu:,r,s] += (Ru[-nu:] if c0==0 else -Ru[-nu:]) - np.log1p(np.exp(Ru[-nu:] if c0==0 else -Ru[-nu:]))
                Gamma[-nu:,r,s] += (R[-nu:] if c1==0 else -R[-nu:]) - np.log1p(np.exp(R[-nu:] if c1==0 else -R[-nu:]))
    for t in range(1, T):
        for r in range(Ns): A[t, :] = maxstar(A[t, :], A[t-1, r] + Gamma[t-1, r, :])
    for t in range(T-2, -1, -1):
        for s in range(Ns): B[t, :] = maxstar(B[t, :], B[t+1, s] + Gamma[t+1, :, s])
    M = A[:, :, np.newaxis] + Gamma + B[:, np.newaxis, :]
    Lp = np.full_like(Ap, -np.inf); Lm = np.full_like(Ap, -np.inf)
    for r in range(Ns):
        for s in range(Ns):
            if r >> 1 == s & (2**(nu-1)-1):
                feedback = (r ^ (r >> 1)) & 1; newbit = (s >> (nu-1)) ^ feedback
                if newbit == 1: Lp = maxstar(Lp, M[:-nu, r, s])
                else: Lm = maxstar(Lm, M[:-nu, r, s])
    return Lm - Lp

def LTE_turbo_decoder(LLRs, transport_size, num_iterations=5):
    K = turbo_len(transport_size) - 4
    Ru1 = np.concatenate([LLRs[0,:K],[LLRs[0,K],LLRs[2,K],LLRs[1,K+1]]])
    R1 = np.concatenate([LLRs[1,:K],[LLRs[1,K],LLRs[0,K+1],LLRs[2,K+1]]])
    Ru2 = np.concatenate([LLRs[0,:K][turbo_permutation(K)],[LLRs[0,K+2],LLRs[2,K+2],LLRs[1,K+3]]])
    R2 = np.concatenate([LLRs[2,:K],[LLRs[1,K+2],LLRs[0,K+3],LLRs[2,K+3]]])
    for R in [Ru1,R1,Ru2,R2]: R[:] = np.clip(R, -512, 512)
    E2 = np.zeros(K)
    for _ in range(num_iterations):
        A1 = np.clip(E2[turbo_permutation_inv(K)], -512, 512)
        L1 = BCJR(Ru1, R1, A1); E1 = L1 - Ru1[:-3] - A1
        A2 = np.clip(E1[turbo_permutation(K)], -512, 512)
        L2 = BCJR(Ru2, R2, A2); E2 = L2 - Ru2[:-3] - A2
    return L2[turbo_permutation_inv(K)]


class CRC24A_Table:
    def __init__(self, crc_poly):
        self._t = np.zeros(256, dtype=np.uint32)
        mask = np.uint32(1 << 23)
        for n in np.arange(256, dtype=np.uint32):
            c = n << 16
            for _ in range(8): c = (crc_poly ^ (c << 1)) if (c & mask) else (c << 1)
            self._t[n] = c & 0xFFFFFF

    def __call__(self, data):
        crc = np.uint32(0)
        for byte in data: crc = ((crc << 8) ^ self._t[((crc >> 16) ^ byte) & 0xFF]) & 0xFFFFFF
        return crc & 0xFFFFFF

In [26]:
class DLSCHDecoding:
    def __init__(self, noise_sigma=0.3, llr_scale=20, num_iterations=5):
        self.noise_sigma = noise_sigma
        self.llr_scale = llr_scale
        self.num_iterations = num_iterations
        self.crc = CRC24A_Table(0x864CFB)

    def __call__(self, chunk):

        pdsch_bits = pdsch_descramble(chunk.pdsch_soft, chunk.ns, chunk.N_id)
        coded_block_size = chunk.tbs + 24 + 4
        pdsch_coded = turbo_rate_dematch(pdsch_bits, coded_block_size, chunk.rv)
        pdsch_LLR = -2 * pdsch_coded / self.noise_sigma**2
        sib1_LLRs = LTE_turbo_decoder(pdsch_LLR * self.llr_scale, chunk.tbs, num_iterations=self.num_iterations)
        hard_bytes = np.packbits(sib1_LLRs < 0)

        if self.crc(hard_bytes) == 0:
            chunk.sib1_decoded = True
            chunk.sib1_bytes = hard_bytes[:-3]
            
        return chunk

In [27]:
class SIB1Chunk:

    def __init__(self, data, tag, N_id, N_rb, ns, f_d, n_ant,
                 phich_res, sfn, n_symbols=14, is_slot_start=True):
        N_sc = N_rb * 12

        # input
        self.data = data
        self.tag = tag
        self.N_id = N_id
        self.N_rb = N_rb
        self.N_sc = N_sc
        self.ns = ns
        self.f_d = f_d
        self.n_symbols = n_symbols
        self.is_slot_start = is_slot_start
        self.n_ant = n_ant
        self.phich_res = phich_res
        self.sfn = sfn

        # CRS outputs
        self.symbols = np.zeros((n_symbols, N_sc), dtype=complex)
        self.H = np.zeros((4, n_symbols, N_sc), dtype=complex)

        # PCFICH outputs
        self.cfi = None
        self.cfi_hamming = None

        # PDCCH outputs
        self.dci_decoded = False
        self.dci_cost = None
        self.rb_start = None
        self.L_crbs = None
        self.Q_m = None
        self.rv = None
        self.tbs = None

        # PDSCH outputs
        self.pdsch_soft = None

        # DLSCH outputs
        self.sib1_decoded = False
        self.sib1_bytes = None

accuracy + time cost for numpy

In [28]:
from data.lte_system_info import LTEParams

In [29]:
rxf = np.load('data/rx_preprocessed.npy')
Fs = float(np.load('data/Fs.npy'))
params = LTEParams(Fs=Fs)

In [30]:
from cell_search import PSSDetection, SSSDetection, PSSChunk
from crs_estimate import CRSChannelEstimation
from mib_decode import PBCHDecoding, BCHDecoding, MIBChunk

In [31]:
N_overlap = params.N_CP + 2 * params.N_FFT
N_subframe = params.N_subframe
stride = N_subframe - N_overlap

pss = PSSDetection(params, peak_ratio=5.0)

pos = 0
chunk_id = 0
pss_count = 0
cells = []

while pos + N_subframe <= len(rxf):

    data = rxf[pos:pos+N_subframe]
    chunk = PSSChunk(data, chunk_id)
    chunk = pss(chunk)

    if chunk.pss_detected:
        pss_count += 1
        global_pss = pos + chunk.pss_local_index
        print(f"Chunk {chunk_id}: PSS detected | "
              f"N_id_2={chunk.N_id_2}, "
              f"local={chunk.pss_local_index}, global={global_pss}")
        cells.append(chunk)    

    pos += stride
    chunk_id += 1

print(f"\nScanned {chunk_id} chunks over {len(rxf)} samples, "
      f"{pss_count} PSS detected")

Chunk 0: PSS detected | N_id_2=2, local=12309, global=12309
Chunk 2: PSS detected | N_id_2=2, local=9563, global=36043
Chunk 6: PSS detected | N_id_2=2, local=9667, global=89107
Chunk 8: PSS detected | N_id_2=2, local=6922, global=112842
Chunk 12: PSS detected | N_id_2=2, local=7027, global=165907
Chunk 13: PSS detected | N_id_2=0, local=11192, global=183312
Chunk 14: PSS detected | N_id_2=2, local=4283, global=189643
Chunk 18: PSS detected | N_id_2=2, local=4390, global=242710
Chunk 19: PSS detected | N_id_2=2, local=14327, global=265887
Chunk 20: PSS detected | N_id_2=2, local=1642, global=266442
Chunk 22: PSS detected | N_id_2=2, local=14330, global=305610
Chunk 24: PSS detected | N_id_2=2, local=1746, global=319506
Chunk 25: PSS detected | N_id_2=2, local=12243, global=343243
Chunk 26: PSS detected | N_id_2=1, local=2587, global=346827
Chunk 29: PSS detected | N_id_2=2, local=12349, global=396309
Chunk 31: PSS detected | N_id_2=2, local=9602, global=420042
Chunk 35: PSS detected | 

In [32]:
sss = SSSDetection(params, peak_ratio=8.0)

sss_cells = []
for chunk in cells:
    chunk = sss(chunk)
    if chunk.sss_detected:
        global_pss = chunk.tag * stride + chunk.pss_local_index
        PCI = 3 * chunk.N_id_1 + chunk.N_id_2
        print(f"Chunk {chunk.tag}: PCI={PCI} | "
              f"N_id_1={chunk.N_id_1}, N_id_2={chunk.N_id_2}, F={chunk.F}, "
              f"local={chunk.pss_local_index}, global={global_pss} | "
              f"f_d={chunk.f_d:.1f}")
        sss_cells.append(chunk)

print(f"\n{len(sss_cells)} SSS detected out of {len(cells)} PSS chunks")

if sss_cells:
    first_global = sss_cells[0].tag * stride + sss_cells[0].pss_local_index
    expected = (len(rxf) - first_global) // params.N_half_frame + 1
    print(f"Expected {expected} cell found ({params.N_half_frame} spacing from {first_global})")

Chunk 2: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=9563, global=36043 | f_d=1126.9
Chunk 8: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=6922, global=112842 | f_d=1128.4
Chunk 14: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=4283, global=189643 | f_d=1164.2
Chunk 20: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=1642, global=266442 | f_d=1204.1
Chunk 25: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=12243, global=343243 | f_d=1180.9
Chunk 31: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=9602, global=420042 | f_d=1107.1
Chunk 37: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=6962, global=496842 | f_d=1179.5
Chunk 43: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=4323, global=573643 | f_d=1142.0
Chunk 49: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=1681, global=650441 | f_d=1269.2
Chunk 54: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=12282, global=727242 | f_d=1165.1
Chunk 60: PCI=380 | N_id_1=126, N_id_2=2, F=0, local=9642, global=804042 | f_d=1122.2
Chunk 66: PCI=380 | N_id_1=126, N_id_2=2, F=1, local=70

In [33]:
crs_np = CRSChannelEstimation(params)
pbch_np = PBCHDecoding()
bch_np = BCHDecoding()

N_id = 3 * sss_cells[0].N_id_1 + sss_cells[0].N_id_2

crs_np.config(N_id=N_id, N_rb=6, ns=1, n_symbols=4, is_slot_start=True)
pbch_np.config(N_id=N_id)
bch_np.config(N_id=N_id)

np_mib_cells = []
for cell in sss_cells:
    if cell.F != 0:
        continue
    f_d = cell.f_d
    pss_global = cell.tag * stride + cell.pss_local_index
    slot1_start = pss_global + params.N_FFT
    data = rxf[slot1_start:slot1_start + params.pbch_len]

    chunk = MIBChunk(data, tag=cell.tag, N_id=N_id, f_d=f_d)
    chunk.pss_global = pss_global  # add this for later reference
    crs_np(chunk)
    for n_ant in [1, 2, 4]:
        chunk.n_ant = n_ant
        pbch_np(chunk)
        bch_np(chunk)
        if chunk.mib_decoded:
            break

    np_mib_cells.append(chunk)
    if chunk.mib_decoded:
        print(f"Chunk {cell.tag}: SFN={chunk.sfn}, BW={chunk.dl_bw}, n_ant={chunk.n_ant}, sec={chunk.sec}, cost={chunk.cost}")
    else:
        print(f"Chunk {cell.tag}: FAILED")

print(f"\n{sum(c.mib_decoded for c in np_mib_cells)}/{len(np_mib_cells)} decoded")

Chunk 2: SFN=313, BW=50, n_ant=2, sec=1, cost=0.0
Chunk 14: SFN=314, BW=50, n_ant=2, sec=2, cost=0.0
Chunk 25: SFN=315, BW=50, n_ant=2, sec=3, cost=0.0
Chunk 37: SFN=316, BW=50, n_ant=2, sec=0, cost=0.0
Chunk 49: SFN=317, BW=50, n_ant=2, sec=1, cost=0.0
Chunk 60: SFN=318, BW=50, n_ant=2, sec=2, cost=0.0
Chunk 72: SFN=319, BW=50, n_ant=2, sec=3, cost=0.0
Chunk 83: SFN=320, BW=50, n_ant=2, sec=0, cost=0.0
Chunk 95: SFN=321, BW=50, n_ant=2, sec=1, cost=0.0
Chunk 107: SFN=322, BW=50, n_ant=2, sec=2, cost=0.0
Chunk 118: SFN=323, BW=50, n_ant=2, sec=3, cost=0.0
Chunk 130: SFN=324, BW=50, n_ant=2, sec=0, cost=0.0
Chunk 141: SFN=325, BW=50, n_ant=2, sec=1, cost=0.0
Chunk 153: SFN=326, BW=50, n_ant=2, sec=2, cost=0.0
Chunk 165: SFN=327, BW=50, n_ant=2, sec=3, cost=0.0
Chunk 176: SFN=328, BW=50, n_ant=2, sec=0, cost=0.0
Chunk 188: SFN=329, BW=50, n_ant=2, sec=1, cost=0.0
Chunk 199: SFN=330, BW=50, n_ant=2, sec=2, cost=0.0
Chunk 211: SFN=331, BW=50, n_ant=2, sec=3, cost=0.0
Chunk 223: SFN=332, BW

In [35]:
crs_np = CRSChannelEstimation(params)
pcfich_np = PCFICHDecoding() 
pdcch_np = PDCCHDecoding()
pdsch_np = PDSCHDecoding() 
dlsch_np = DLSCHDecoding(num_iterations=5)

N_rb = np_mib_cells[0].dl_bw 
n_ant = np_mib_cells[0].n_ant 
phich_res = np_mib_cells[0].phich_res

crs_np.config(N_id=N_id, N_rb=N_rb, ns=10, n_symbols=14, is_slot_start=True)
pcfich_np.config(N_id=N_id, N_rb=N_rb, ns=10)
pdcch_np.config(N_id=N_id, N_rb=N_rb, n_ant=n_ant, phich_res=phich_res, ns=10)
pdsch_np.config(N_id=N_id, n_ant=n_ant)

np_sib1_cells = []
n_dci_fail = 0
n_sib1_fail = 0

for cell in np_mib_cells:
    if cell.sfn % 2 != 0: continue
    p = params
    slot1_start = cell.pss_global + p.N_FFT    
    frame_start = slot1_start - p.N_slot
    subf5_start = frame_start + 5 * p.N_subframe
    data = rxf[subf5_start:subf5_start + p.N_subframe]

    chunk = SIB1Chunk(data=data, tag=cell.sfn, N_id=N_id, N_rb=N_rb,
        ns=10, f_d=cell.f_d, n_ant=n_ant, phich_res=phich_res, sfn=cell.sfn)
    
    crs_np(chunk) 
    pcfich_np(chunk) 
    pdcch_np(chunk)

    if not chunk.dci_decoded:
        n_dci_fail += 1
        continue

    pdsch_np(chunk)
    dlsch_np(chunk)
    np_sib1_cells.append(chunk)

    if not chunk.sib1_decoded:
        n_sib1_fail += 1

# ---- summary ----
n_even = sum(1 for c in np_mib_cells if c.sfn % 2 == 0)
n_ok = sum(c.sib1_decoded for c in np_sib1_cells)
print(f'=== SIB1 Decode Summary ===')
print(f'Even frames: {n_even}, DCI decoded: {n_even - n_dci_fail}, DCI failed: {n_dci_fail}')
print(f'SIB1 decoded: {n_ok}, SIB1 failed: {n_sib1_fail}')

# ---- per-frame details ----
print(f'\n{"SFN":>5s}  {"CFI":>3s}  {"hamm":>4s}  {"DCI":>4s}  {"rb":>3s}  {"L":>3s}  {"Qm":>3s}  {"rv":>3s}  {"TBS":>4s}  {"SIB1":>5s}')
print('-' * 55)
for chunk in np_sib1_cells:
    print(f'{chunk.sfn:>5d}  {chunk.cfi:>3d}  {chunk.cfi_hamming:>4d}  '
          f'{"ok":>4s}  {chunk.rb_start:>3d}  {chunk.L_crbs:>3d}  {chunk.Q_m:>3d}  '
          f'{chunk.rv:>3d}  {chunk.tbs:>4d}  '
          f'{"ok" if chunk.sib1_decoded else "FAIL":>5s}')

=== SIB1 Decode Summary ===
Even frames: 50, DCI decoded: 50, DCI failed: 0
SIB1 decoded: 50, SIB1 failed: 0

  SFN  CFI  hamm   DCI   rb    L   Qm   rv   TBS   SIB1
-------------------------------------------------------
  314    1     0    ok    0    4    2    2   176     ok
  316    1     0    ok    0    4    2    3   176     ok
  318    1     0    ok    0    4    2    1   176     ok
  320    1     0    ok    0    4    2    0   176     ok
  322    1     0    ok    0    4    2    2   176     ok
  324    1     0    ok    0    4    2    3   176     ok
  326    2     0    ok    0    4    2    1   176     ok
  328    2     0    ok    0    4    2    0   176     ok
  330    2     0    ok    0    4    2    2   176     ok
  332    2     0    ok    0    4    2    3   176     ok
  334    2     0    ok    0    4    2    1   176     ok
  336    2     0    ok    0    4    2    0   176     ok
  338    2     0    ok    0    4    2    2   176     ok
  340    2     0    ok    0    4    2    3   176  

In [18]:
# 1. unit time
pcfich_time = time_stage(pcfich_np, np_sib1_cells[0])
print(f"PCFICH unit time: {pcfich_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(pcfich_np, np_sib1_cells)

print(f"PCFICH concurrency test: {len(np_sib1_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(np_sib1_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

PCFICH unit time: 0.03 ms

PCFICH concurrency test: 50 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        0.10     1.00x
       2     2        0.11     0.89x
       4     4        0.13     0.76x
       6     6        0.16     0.61x
      10    10        0.22     0.45x


In [25]:
# 1. unit time
pdcch_time = time_stage(pdcch_np, np_sib1_cells[0])
print(f"PDCCH unit time: {pdcch_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(pdcch_np, np_sib1_cells)

print(f"PDCCH concurrency test: {len(np_sib1_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(np_sib1_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

PDCCH unit time: 0.09 ms

PDCCH concurrency test: 50 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        0.26     1.00x
       2     2        0.24     1.09x
       4     4        0.27     0.96x
       6     6        0.29     0.91x
      10    10        0.33     0.78x


In [30]:
# 1. unit time
pdsch_time = time_stage(pdsch_np, np_sib1_cells[0])
print(f"PDSCH unit time: {pdsch_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(pdsch_np, np_sib1_cells)

print(f"PDSCH concurrency test: {len(np_sib1_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(np_sib1_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

PDSCH unit time: 0.06 ms

PDSCH concurrency test: 50 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        0.13     1.00x
       2     2        0.19     0.69x
       4     4        0.25     0.53x
       6     6        0.32     0.41x
      10    10        0.32     0.40x


In [31]:
# 1. unit time
dlsch_time = time_stage(dlsch_np, np_sib1_cells[0])
print(f"DLSCH unit time: {dlsch_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(dlsch_np, np_sib1_cells)

print(f"DLSCH concurrency test: {len(np_sib1_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(np_sib1_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

DLSCH unit time: 232.71 ms

DLSCH concurrency test: 50 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1      224.72     1.00x
       2     2      208.17     1.08x
       4     4      214.38     1.05x
       6     6      218.85     1.03x
      10    10      229.53     0.98x


2. numba

In [35]:
@numba.jit(nopython=True, nogil=True)
def _log_sigmoid(x):
    if x >= 0.0:
        return -np.log1p(np.exp(-x))
    else:
        return x - np.log1p(np.exp(x))


@numba.jit(nopython=True, nogil=True)
def _maxstar_nb(a, b):
    if a <= -1e9:
        return b
    if b <= -1e9:
        return a
    mx = a if a > b else b
    return mx + np.log1p(np.exp(-abs(a - b)))


@numba.jit(nopython=True, nogil=True)
def _bcjr_numba(Ru, R, Ap,
                nt_edges, nt_sign_ap, nt_sign_c0, nt_sign_c1,
                term_edges, term_sign_c0, term_sign_c1,
                bit1_edges, bit0_edges):
    """Full BCJR pass. GIL released.

    nt_edges:  (N_nt, 2) int — (r, s) pairs for non-termination edges
    nt_sign_*: (N_nt,) float — sign multipliers per edge
    term_edges: (N_term, 2) int
    bit1_edges, bit0_edges: (N_b1, 2), (N_b0, 2) — edges where info bit is 1 or 0
    """
    K = Ap.shape[0]
    T = Ru.shape[0]
    Ns = 8
    NEG_INF = -1e10
    N_nt = nt_edges.shape[0]
    N_term = term_edges.shape[0]

    # Gamma
    gamma = np.full((T, Ns, Ns), NEG_INF)

    for t in range(K):
        for e in range(N_nt):
            r = nt_edges[e, 0]; s = nt_edges[e, 1]
            gamma[t, r, s] = (_log_sigmoid(Ap[t] * nt_sign_ap[e])
                            + _log_sigmoid(Ru[t] * nt_sign_c0[e])
                            + _log_sigmoid(R[t] * nt_sign_c1[e]))

    for t in range(K, T):
        for e in range(N_term):
            r = term_edges[e, 0]; s = term_edges[e, 1]
            gamma[t, r, s] = (_log_sigmoid(Ru[t] * term_sign_c0[e])
                            + _log_sigmoid(R[t] * term_sign_c1[e]))

    # Forward
    alpha = np.full((T + 1, Ns), NEG_INF)
    alpha[0, 0] = 0.0
    for t in range(T):
        for s in range(Ns):
            val = NEG_INF
            for r in range(Ns):
                val = _maxstar_nb(val, alpha[t, r] + gamma[t, r, s])
            alpha[t + 1, s] = val

    # Backward
    beta = np.full((T + 1, Ns), NEG_INF)
    beta[T, 0] = 0.0
    for t in range(T - 1, -1, -1):
        for r in range(Ns):
            val = NEG_INF
            for s in range(Ns):
                val = _maxstar_nb(val, beta[t + 1, s] + gamma[t, r, s])
            beta[t, r] = val

    # LLR
    N_b1 = bit1_edges.shape[0]
    N_b0 = bit0_edges.shape[0]
    Lp = np.full(K, NEG_INF)
    Lm = np.full(K, NEG_INF)

    for t in range(K):
        for e in range(N_b1):
            r = bit1_edges[e, 0]; s = bit1_edges[e, 1]
            m = alpha[t, r] + gamma[t, r, s] + beta[t + 1, s]
            Lp[t] = _maxstar_nb(Lp[t], m)
        for e in range(N_b0):
            r = bit0_edges[e, 0]; s = bit0_edges[e, 1]
            m = alpha[t, r] + gamma[t, r, s] + beta[t + 1, s]
            Lm[t] = _maxstar_nb(Lm[t], m)

    result = np.empty(K)
    for i in range(K):
        result[i] = Lm[i] - Lp[i]
    return result

def _build_trellis_edges():
    """Build edge-list trellis for Numba (compact, no 8×8 matrices)."""
    Ns = 8; nu = 3
    nt_list = []; nt_sap = []; nt_sc0 = []; nt_sc1 = []
    bit1_list = []; bit0_list = []
    term_list = []; term_sc0 = []; term_sc1 = []

    for r in range(Ns):
        for s in range(Ns):
            if (r >> 1) == (s & (2**(nu-1)-1)):
                feedback = (r ^ (r >> 1)) & 1
                newbit = (s >> (nu-1)) ^ feedback
                c0 = newbit; c1 = (r ^ (r >> 2) ^ (s >> (nu-1))) & 1
                nt_list.append([r, s])
                nt_sap.append(1.0 - 2.0 * newbit)
                nt_sc0.append(1.0 - 2.0 * c0)
                nt_sc1.append(1.0 - 2.0 * c1)
                if newbit == 1: bit1_list.append([r, s])
                else: bit0_list.append([r, s])
            if r >> 1 == s:
                feedback = (r ^ (r >> 1)) & 1
                c0 = feedback; c1 = (r ^ (r >> 2)) & 1
                term_list.append([r, s])
                term_sc0.append(1.0 - 2.0 * c0)
                term_sc1.append(1.0 - 2.0 * c1)

    return (np.array(nt_list, dtype=np.int64), np.array(nt_sap), np.array(nt_sc0), np.array(nt_sc1),
            np.array(term_list, dtype=np.int64), np.array(term_sc0), np.array(term_sc1),
            np.array(bit1_list, dtype=np.int64), np.array(bit0_list, dtype=np.int64))


@numba.jit(nopython=True, nogil=True)
def _c_sequence_nb(M, c_init):
    """Gold sequence generator. Replaces Python c_sequence."""
    Nc = 1600
    x_1 = np.zeros(M + Nc, dtype=np.uint8)
    x_1[0] = 1
    for n in range(31, M + Nc):
        x_1[n] = x_1[n - 28] ^ x_1[n - 31]

    x_2 = np.zeros(M + Nc, dtype=np.uint8)
    for n in range(31):
        x_2[n] = np.uint8((c_init >> n) & 1)
    for n in range(31, M + Nc):
        x_2[n] = x_2[n - 28] ^ x_2[n - 29] ^ x_2[n - 30] ^ x_2[n - 31]

    result = np.empty(M, dtype=np.uint8)
    for i in range(M):
        result[i] = x_1[Nc + i] ^ x_2[Nc + i]
    return result


@numba.jit(nopython=True, nogil=True)
def _turbo_perm_nb(K, f1, f2):
    """Turbo code interleaver permutation."""
    perm = np.empty(K, dtype=np.int64)
    for i in range(K):
        perm[i] = (f1 * i + f2 * i * i) % K
    return perm


@numba.jit(nopython=True, nogil=True)
def _invert_perm_nb(perm):
    """Invert a permutation array. O(n)."""
    n = len(perm)
    inv = np.empty(n, dtype=np.int64)
    for i in range(n):
        inv[perm[i]] = i
    return inv


@numba.jit(nopython=True, nogil=True)
def _rate_matching_turbo_nb(seq_len, col_perm, rv):
    """Turbo code rate matching interleaver. Replaces Python rate_matching_turbo."""
    C = 32
    D = seq_len
    DUMMY = D + 10000

    R = (D + C - 1) // C
    K_pi = R * C
    N_dummy = K_pi - D

    # build y: dummy symbols + data indices
    y = np.empty(K_pi, dtype=np.int64)
    for i in range(N_dummy):
        y[i] = DUMMY
    for i in range(D):
        y[N_dummy + i] = i

    # column permutation (ports 1 & 2)
    v = np.empty(K_pi, dtype=np.int64)
    for col in range(C):
        src_col = col_perm[col]
        for row in range(R):
            v[col * R + row] = y[row * C + src_col]

    # port 3: QPP interleaver
    v3 = np.empty(K_pi, dtype=np.int64)
    for k in range(K_pi):
        pi_k = (col_perm[k // R] + C * (k % R) + 1) % K_pi
        v3[k] = y[pi_k]

    # offset streams
    v1 = v.copy()
    v2 = v.copy()
    for i in range(K_pi):
        if v2[i] < 10000:
            v2[i] += D
        if v3[i] < 10000:
            v3[i] += 2 * D

    # combine circular buffer
    N_cb = 3 * K_pi
    v_comb = np.zeros(N_cb, dtype=np.int64)
    for i in range(K_pi):
        v_comb[i] = v1[i]
    for i in range(K_pi):
        v_comb[K_pi + 2 * i] = v2[i]
        v_comb[K_pi + 2 * i + 1] = v3[i]

    # RV-dependent start and selection
    k0 = R * (2 * int(np.ceil(N_cb / (8.0 * R))) * rv + 2)

    e = np.empty(3 * D, dtype=np.int64)
    k = 0
    j = 0
    while k < 3 * D:
        idx = (k0 + j) % N_cb
        if v_comb[idx] != DUMMY:
            e[k] = v_comb[idx]
            k += 1
        j += 1

    return e


@numba.jit(nopython=True, nogil=True)
def _crc24a_nb(data, crc_table):
    """CRC-24A check on packed bytes."""
    crc = np.uint32(0)
    for i in range(len(data)):
        idx = np.uint32(((crc >> 16) ^ np.uint32(data[i])) & np.uint32(0xFF))
        crc = ((crc << 8) ^ crc_table[idx]) & np.uint32(0xFFFFFF)
    return crc


@numba.jit(nopython=True, nogil=True)
def _packbits_nb(bits, n_bytes):
    """Pack boolean/uint8 bit array to bytes."""
    result = np.zeros(n_bytes, dtype=np.uint8)
    for i in range(n_bytes):
        val = np.uint8(0)
        for j in range(8):
            idx = i * 8 + j
            if idx < len(bits):
                val = (val << np.uint8(1)) | np.uint8(bits[idx])
            else:
                val = val << np.uint8(1)
        result[i] = val
    return result


# ---- Main DLSCH function: everything in one nogil call ----

@numba.jit(nopython=True, nogil=True)
def _dlsch_full_nb(pdsch_soft, c_init_descramble,
                   tbs, rv, f1, f2,
                   noise_sigma_inv2, llr_scale, num_iterations,
                   col_perm_rm,
                   nt_edges, nt_sign_ap, nt_sign_c0, nt_sign_c1,
                   term_edges, term_sign_c0, term_sign_c1,
                   bit1_edges, bit0_edges,
                   maxR, maxA,
                   crc24_table):
    """Full DLSCH decode: descramble → rate dematch → turbo → CRC. GIL released."""

    n_soft = len(pdsch_soft)

    # ---- 1. PDSCH descramble ----
    c = _c_sequence_nb(n_soft, c_init_descramble)
    descrambled = np.empty(n_soft)
    for i in range(n_soft):
        descrambled[i] = pdsch_soft[i] * (1.0 - 2.0 * c[i])

    # ---- 2. Turbo rate dematch ----
    coded_block_size = tbs + 24 + 4
    collect_len = 3 * coded_block_size

    perm_rm = _rate_matching_turbo_nb(coded_block_size, col_perm_rm, rv)

    coded_flat = np.zeros(collect_len)
    for i in range(collect_len):
        if i < n_soft:
            coded_flat[perm_rm[i]] = descrambled[i]

    # reshape to (3, coded_block_size)
    coded = np.empty((3, coded_block_size))
    for s in range(3):
        for j in range(coded_block_size):
            coded[s, j] = coded_flat[s * coded_block_size + j]

    # ---- 3. LLR scaling ----
    LLRs = np.empty((3, coded_block_size))
    for s in range(3):
        for j in range(coded_block_size):
            LLRs[s, j] = coded[s, j] * noise_sigma_inv2 * llr_scale

    # ---- 4. Turbo decode ----
    K = tbs + 24  # without trellis termination

    perm = _turbo_perm_nb(K, f1, f2)
    perm_inv = _invert_perm_nb(perm)

    # trellis termination shuffling (TS 36.212 §5.1.3.2.2)
    Ru1 = np.empty(K + 3)
    R1 = np.empty(K + 3)
    Ru2 = np.empty(K + 3)
    R2 = np.empty(K + 3)

    for i in range(K):
        Ru1[i] = LLRs[0, i]
        R1[i] = LLRs[1, i]
        Ru2[i] = LLRs[0, perm[i]]
        R2[i] = LLRs[2, i]

    Ru1[K] = LLRs[0, K];     Ru1[K+1] = LLRs[2, K];     Ru1[K+2] = LLRs[1, K+1]
    R1[K] = LLRs[1, K];      R1[K+1] = LLRs[0, K+1];    R1[K+2] = LLRs[2, K+1]
    Ru2[K] = LLRs[0, K+2];   Ru2[K+1] = LLRs[2, K+2];   Ru2[K+2] = LLRs[1, K+3]
    R2[K] = LLRs[1, K+2];    R2[K+1] = LLRs[0, K+3];    R2[K+2] = LLRs[2, K+3]

    # clip
    for i in range(K + 3):
        if Ru1[i] > maxR: Ru1[i] = maxR
        elif Ru1[i] < -maxR: Ru1[i] = -maxR
        if R1[i] > maxR: R1[i] = maxR
        elif R1[i] < -maxR: R1[i] = -maxR
        if Ru2[i] > maxR: Ru2[i] = maxR
        elif Ru2[i] < -maxR: Ru2[i] = -maxR
        if R2[i] > maxR: R2[i] = maxR
        elif R2[i] < -maxR: R2[i] = -maxR

    # turbo iterations
    E2 = np.zeros(K)
    L2 = np.zeros(K)

    for iteration in range(num_iterations):
        A1 = np.empty(K)
        for i in range(K):
            v = E2[perm_inv[i]]
            if v > maxA: v = maxA
            elif v < -maxA: v = -maxA
            A1[i] = v

        L1 = _bcjr_numba(Ru1, R1, A1,
                         nt_edges, nt_sign_ap, nt_sign_c0, nt_sign_c1,
                         term_edges, term_sign_c0, term_sign_c1,
                         bit1_edges, bit0_edges)

        E1 = np.empty(K)
        for i in range(K):
            E1[i] = L1[i] - Ru1[i] - A1[i]

        A2 = np.empty(K)
        for i in range(K):
            v = E1[perm[i]]
            if v > maxA: v = maxA
            elif v < -maxA: v = -maxA
            A2[i] = v

        L2 = _bcjr_numba(Ru2, R2, A2,
                         nt_edges, nt_sign_ap, nt_sign_c0, nt_sign_c1,
                         term_edges, term_sign_c0, term_sign_c1,
                         bit1_edges, bit0_edges)

        for i in range(K):
            E2[i] = L2[i] - Ru2[i] - A2[i]

    # de-interleave
    sib1_LLRs = np.empty(K)
    for i in range(K):
        sib1_LLRs[i] = L2[perm_inv[i]]

    # ---- 5. Hard decision ----
    n_bits = tbs + 24  # K without trellis
    hard_bits = np.empty(n_bits, dtype=np.uint8)
    for i in range(n_bits):
        hard_bits[i] = np.uint8(1) if sib1_LLRs[i] < 0 else np.uint8(0)

    # ---- 6. Pack bits + CRC-24A ----
    n_bytes = (n_bits + 7) // 8
    hard_bytes = _packbits_nb(hard_bits, n_bytes)

    crc_ok = _crc24a_nb(hard_bytes, crc24_table) == np.uint32(0)

    # return payload bytes (exclude 3-byte CRC)
    payload = hard_bytes[:n_bytes - 3]
    return crc_ok, payload

In [36]:
class DLSCHDecodingNumba:

    COL_PERM_RM = np.array([0,16,8,24,4,20,12,28,2,18,10,26,
                            6,22,14,30,1,17,9,25,5,21,13,29,
                            3,19,11,27,7,23,15,31], dtype=np.int64)

    def __init__(self, noise_sigma=0.3, llr_scale=20, num_iterations=5):
        self.noise_sigma_inv2 = -2.0 / noise_sigma**2
        self.llr_scale = float(llr_scale)
        self.num_iterations = num_iterations

        # trellis tables
        tables = _build_trellis_edges()
        (self._nt_edges, self._nt_sap, self._nt_sc0, self._nt_sc1,
         self._term_edges, self._term_sc0, self._term_sc1,
         self._bit1_edges, self._bit0_edges) = tables

        # CRC-24A table
        crc_obj = CRC24A_Table(0x864CFB)
        self._crc24_table = crc_obj._t.astype(np.uint32)

        self._c_init = None

        # warmup
        dummy_soft = np.zeros(100)
        f1, f2 = f1f2_table(40)
        _dlsch_full_nb(dummy_soft, 0, 16, 0, f1, f2,
                       self.noise_sigma_inv2, self.llr_scale, 1,
                       self.COL_PERM_RM,
                       self._nt_edges, self._nt_sap, self._nt_sc0, self._nt_sc1,
                       self._term_edges, self._term_sc0, self._term_sc1,
                       self._bit1_edges, self._bit0_edges,
                       512.0, 512.0, self._crc24_table)

    def config(self, ns, N_id):
        """Precompute descrambling c_init."""
        self._c_init = (0xFFFF << 14) + ((ns // 2) << 9) + N_id

    def __call__(self, chunk):
        chunk.sib1_decoded = False
        chunk.sib1_bytes = None

        K = chunk.tbs + 24
        f1, f2 = f1f2_table(K)

        crc_ok, payload = _dlsch_full_nb(
            chunk.pdsch_soft, self._c_init,
            chunk.tbs, chunk.rv, f1, f2,
            self.noise_sigma_inv2, self.llr_scale, self.num_iterations,
            self.COL_PERM_RM,
            self._nt_edges, self._nt_sap, self._nt_sc0, self._nt_sc1,
            self._term_edges, self._term_sc0, self._term_sc1,
            self._bit1_edges, self._bit0_edges,
            512.0, 512.0, self._crc24_table)

        if crc_ok:
            chunk.sib1_decoded = True
            chunk.sib1_bytes = payload

        return chunk

In [ ]:
dlsch_nb = DLSCHDecodingNumba(num_iterations=5)

dlsch_nb.config(ns=10, N_id=N_id)  # precomputes c_init for descrambling

nb_sib1_cells = []
for cell in np_mib_cells:
    if cell.sfn % 2 != 0: continue
    p = params
    slot1_start = cell.pss_global + p.N_FFT    
    frame_start = slot1_start - p.N_slot
    subf5_start = frame_start + 5 * p.N_subframe
    data = rxf[subf5_start:subf5_start + p.N_subframe]

    chunk = SIB1Chunk(data=data, tag=cell.sfn, N_id=N_id, N_rb=N_rb,
        ns=10, f_d=cell.f_d, n_ant=n_ant, phich_res=phich_res, sfn=cell.sfn)
    
    crs_np(chunk) 
    pcfich_np(chunk) 
    pdcch_np(chunk)
    if chunk.dci_decoded: 
        pdsch_np(chunk) 
        dlsch_np(chunk)
        nb_sib1_cells.append(chunk)

print(f'{sum(c.sib1_decoded for c in nb_sib1_cells)}/{len(nb_sib1_cells)} SIB1 decoded')

50/50 SIB1 decoded


In [39]:
# 1. unit time
dlsch_time = time_stage(dlsch_nb, nb_sib1_cells[0])
print(f"DLSCH unit time: {dlsch_time:.2f} ms\n")

# 2. concurrency
results = test_concurrency(dlsch_nb, nb_sib1_cells)

print(f"DLSCH concurrency test: {len(nb_sib1_cells)} chunks\n")
print(f"{'workers':>8s}  {'cc':>4s}  {'unit(ms)':>10s}  {'speedup':>8s}")
print("-" * 37)
baseline = results[0][2]
n = len(nb_sib1_cells)
for workers, cc, elapsed in results:
    unit_ms = elapsed / n * 1000
    print(f"{workers:>8d}  {cc:>4d}  {unit_ms:>10.2f}  {baseline/elapsed:>7.2f}x")

DLSCH unit time: 2.31 ms

DLSCH concurrency test: 50 chunks

 workers    cc    unit(ms)   speedup
-------------------------------------
       1     1        2.33     1.00x
       2     2        1.31     1.79x
       4     4        0.62     3.78x
       6     6        0.50     4.63x
      10    10        0.38     6.14x
